# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This week's job: check two signals the rule
leans on, encode one transparent rule from them, and hand-review the top 10 with a skeptic's eye.
This baseline is what the Week-5 model has to beat.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'{len(df):,} rows, {df["client_id"].nunique()} clients')


30,000 rows, 32 clients


## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth a refresh review if it's still visible, it hasn't
been touched in a while, it's sitting in a position where clicks should be happening, and its
actual CTR is falling well short of what pages at that position normally earn. The size of that
shortfall — scaled by how much traffic is riding on it — is the score.

**Reason code (one, constant when the rule fires):** `ctr_underperforms_position_and_stale`

**Two signals this rule leans on, checked first, each with a bucket table and n:**

1. **Staleness -> decline** (`days_since_last_update` / `freshness_tier`) — this is the signal
   behind FlyRank's refresh flags (the 180-day `stale_visible_page` threshold from the session).
2. **CTR vs. position** (`ctr` / `position_tier`) — this is the signal behind the CTR-fix logic
   (the `low_ctr_visible_page` flag only makes sense relative to what a page's position should
   earn).

Both checks use a **volume floor of n >= 500 per bucket** before a bucket counts toward the
verdict — the data dictionary flags exactly this risk for `position_tier` medians, and it turned
out to matter for `freshness_tier` too (see below).


In [2]:
# Signal 1: staleness vs. decline rate (freshness_tier), volume floor n >= 500
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)  # evaluation-only, never a rule input

sig1 = (df.groupby('freshness_tier')
          .agg(n=('content_id', 'count'), decline_rate=('is_declining', 'mean'))
          .reindex(['0-30', '31-90', '91-180', '181+']))
print('Signal 1 - freshness_tier vs. decline_rate:')
print(sig1.round(4))

FLOOR = 500
trusted1 = sig1[sig1['n'] >= FLOOR]
print(f'\nTrusted buckets (n >= {FLOOR}):')
print(trusted1.round(4))


Signal 1 - freshness_tier vs. decline_rate:
                    n  decline_rate
freshness_tier                     
0-30            20480        0.5114
31-90             175        0.5886
91-180           9171        0.6111
181+              174        0.4713

Trusted buckets (n >= 500):
                    n  decline_rate
freshness_tier                     
0-30            20480        0.5114
91-180           9171        0.6111


In [3]:
# Signal 2: CTR vs. position_tier, same volume floor, min-visibility filter (matches the
# data dictionary's warning: never read a position_tier stat without a volume floor)
sig2 = (df[df['impressions_90d'] >= 100]
          .groupby('position_tier')
          .agg(n=('content_id', 'count'), mean_ctr=('ctr', 'mean'))
          .reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep']))
print('Signal 2 - position_tier vs. mean_ctr:')
print(sig2.round(4))

trusted2 = sig2[sig2['n'] >= FLOOR]
print(f'\nTrusted buckets (n >= {FLOOR}):')
print(trusted2.round(4))


Signal 2 - position_tier vs. mean_ctr:
                  n  mean_ctr
position_tier                
top_3           533    0.3341
page_1         8633    0.3548
striking       5903    0.2558
page_3_5       6058    0.1424
deep            879    0.0554

Trusted buckets (n >= 500):
                  n  mean_ctr
position_tier                
top_3           533    0.3341
page_1         8633    0.3548
striking       5903    0.2558
page_3_5       6058    0.1424
deep            879    0.0554


**Signal 1 verdict: CONFIRMED.** On the two trustworthy buckets (n=20,480 and n=9,171),
decline rate rises from 51.1% (0-30 days stale) to 61.1% (91-180 days stale) — a real ~10pp gap
on large n. The `31-90` and `181+` buckets (n=175, n=174) are excluded by the floor, and it's a
good thing: `181+` alone would have shown decline *dropping* to 47.1%, which would have wrongly
read as a contradiction if I hadn't set the floor before looking. One consequence worth stating
plainly: this data doesn't support trusting the product's own 180-day `stale_visible_page`
threshold — the 180+ population here is too thin to say anything about it. My rule uses a
**90-day** threshold instead, where the signal is actually confirmed.

**Signal 2 verdict: CONFIRMED.** CTR drops sharply and monotonically from `page_1` (0.355%) to
`deep` (0.055%) — a 6x spread — across three large, well-separated buckets. The one wrinkle:
`top_3` (0.334%) sits slightly *below* `page_1` (0.355%), but at n=533 that 2pp gap is within
plausible sampling noise, not a real inversion, so it doesn't change how the rule uses position.


## 2. Build the ranked queue (writes the CSV)

The rule, coded as simple products of interpretable conditions — no fitted weights, per the
`building-baselines` skill. `ctr_benchmark` comes straight from the Signal 2 bucket table above
(a data-derived lookup, not a model). Every input is available *before* any 30-day trend window
closes: `days_since_last_update`, `impressions_90d`, `position_tier`, `ctr` are all point-in-time
facts about the page today, not comparisons across time.


In [4]:
import os

# Benchmark CTR per position tier, taken directly from the Signal 2 table above (data-derived,
# not fitted).
ctr_benchmark = sig2['mean_ctr'].to_dict()

stale        = (df['days_since_last_update'] >= 90).astype(int)          # from Signal 1's confirmed threshold
visible      = (df['impressions_90d'] >= 100).astype(int)                # minimum-volume floor
position_ok  = df['position_tier'].isin(['top_3', 'page_1', 'striking', 'page_3_5']).astype(int)  # excludes 'deep' and 'no_data'

benchmark = df['position_tier'].map(ctr_benchmark).fillna(0)
ctr_gap   = (benchmark - df['ctr']).clip(lower=0)                        # only an unclaimed gap counts

score = stale * visible * position_ok * ctr_gap * np.log1p(df['impressions_90d'])

queue = df[['content_id', 'client_id', 'impressions_90d', 'avg_position', 'position_tier',
            'ctr', 'days_since_last_update', 'freshness_tier']].copy()
queue['ctr_benchmark'] = benchmark.round(4)
queue['ctr_gap']       = ctr_gap.round(4)
queue['score']         = score.round(2)
queue['reason_code']   = np.where(queue['score'] > 0, 'ctr_underperforms_position_and_stale', 'not_flagged')

# Action label: top 10% of flagged (score > 0) rows get priority, the rest get monitored.
flagged = queue[queue['score'] > 0]
cut = flagged['score'].quantile(0.90)
queue['action'] = 'no_action'
queue.loc[queue['score'] > 0, 'action'] = np.where(flagged['score'] >= cut, 'refresh_priority', 'monitor')

queue = queue.sort_values('score', ascending=False).reset_index(drop=True)

print(f"flagged (score > 0): {(queue['score'] > 0).sum():,} / {len(queue):,}")
print(queue['action'].value_counts())

os.makedirs('../outputs', exist_ok=True)
queue.to_csv('../outputs/baseline_action_score.csv', index=False)
print('\nwrote work/outputs/baseline_action_score.csv')


flagged (score > 0): 5,367 / 30,000
action
no_action           24633
monitor              4821
refresh_priority      546
Name: count, dtype: int64



wrote work/outputs/baseline_action_score.csv


## 3. Top-10 review

One line each: the action, why it's there, what would make it wrong.


In [5]:
top10 = queue.head(10)
top10


,content_id,client_id,impressions_90d,avg_position,position_tier,ctr,days_since_last_update,freshness_tier,ctr_benchmark,ctr_gap,score,reason_code,action
0,content_c8e9d6ab9013,client_19581e27de,208678,9.7,page_1,0.00,104,91-180,0.3548,0.3548,4.35,ctr_underperforms_position_and_stale,refresh_priority
1,content_36ff89c8214e,client_19581e27de,295097,7.3,page_1,0.05,104,91-180,0.3548,0.3048,3.84,ctr_underperforms_position_and_stale,refresh_priority
2,content_c1fe78bc4e37,client_19581e27de,134055,7.5,page_1,0.03,104,91-180,0.3548,0.3248,3.83,ctr_underperforms_position_and_stale,refresh_priority
3,content_4a6607efcb46,client_6208ef0f77,128068,2.2,top_3,0.01,104,91-180,0.3341,0.3241,3.81,ctr_underperforms_position_and_stale,refresh_priority
4,content_b115f7c74779,client_19581e27de,123469,8.0,page_1,0.03,104,91-180,0.3548,0.3248,3.81,ctr_underperforms_position_and_stale,refresh_priority
5,content_d0cc5baa4995,client_19581e27de,83651,6.6,page_1,0.03,104,91-180,0.3548,0.3248,3.68,ctr_underperforms_position_and_stale,refresh_priority
6,content_d07ea098353c,client_19581e27de,63366,9.4,page_1,0.03,104,91-180,0.3548,0.3248,3.59,ctr_underperforms_position_and_stale,refresh_priority
7,content_9c8299b55f3c,client_624b60c58c,54783,8.5,page_1,0.03,104,91-180,0.3548,0.3248,3.54,ctr_underperforms_position_and_stale,refresh_priority
8,content_91652435f57a,client_19581e27de,159590,7.8,page_1,0.06,104,91-180,0.3548,0.2948,3.53,ctr_underperforms_position_and_stale,refresh_priority
9,content_dd635253d90e,client_6208ef0f77,33286,4.6,page_1,0.02,104,91-180,0.3548,0.3348,3.49,ctr_underperforms_position_and_stale,refresh_priority


1. **`content_c8e9d6ab9013`** — `refresh_priority`. 208,678 impressions at position 9.7
   (`page_1`) with a **0.00% CTR** — the single largest unclaimed gap in the dataset. *Wrong if:*
   `avg_position` is stale/mismeasured for this page (e.g. GSC attributing a redirect or a
   near-duplicate URL here), since a true 0.00% CTR at page-1 volume this large is unusual enough
   to double-check before acting.
2. **`content_36ff89c8214e`** — `refresh_priority`. 295k impressions, position 7.3, CTR 0.05%
   against a 0.355% benchmark. *Wrong if:* this page's actual intent is informational/reference
   (people scan the snippet and don't need to click) rather than a page that should be
   converting clicks — CTR gap alone can't tell those apart.
3. **`content_c1fe78bc4e37`** — `refresh_priority`. Same client, same pattern: 134k impressions,
   position 7.5, CTR 0.03%. *Wrong if:* something client-side broke tracking for this client
   around the same window — three of their pages showing near-zero CTR together looks like a
   real content problem, but could also be a tagging issue.
4. **`content_4a6607efcb46`** — `refresh_priority`. Different client, `top_3` position (2.2!)
   with CTR 0.01% against a 0.334% benchmark — a top-3 ranking essentially getting no clicks is
   the sharpest anomaly in the top 10. *Wrong if:* the ranking keyword doesn't match user intent
   (e.g. ranks top-3 for a term nobody actually wants to click through on).
5. **`content_b115f7c74779`** — `refresh_priority`. 123k impressions, position 8.0, CTR 0.03%.
   Same client as #2/#3. *Wrong if:* — see #3; a fourth page from the same client reinforces the
   tracking-issue possibility as much as it reinforces "this client needs a refresh pass."
6. **`content_d0cc5baa4995`** — `refresh_priority`. 83.7k impressions, position 6.6, CTR 0.03%.
   *Wrong if:* the title/meta shown in search doesn't match what's actually on the page anymore —
   a refresh wouldn't fix a mismatch that started upstream of the content itself.
7. **`content_d07ea098353c`** — `refresh_priority`. 63.4k impressions, position 9.4, CTR 0.03%.
   *Wrong if:* seasonal — a page that used to convert well but is being shown for an
   off-season query cluster right now.
8. **`content_9c8299b55f3c`** — `refresh_priority`. 54.8k impressions, position 8.5, CTR 0.03%,
   different client than #2-#7. *Wrong if:* same caveats as the others — position/CTR alone
   can't distinguish "needs a refresh" from "wrong page ranking for this query."
9. **`content_91652435f57a`** — `refresh_priority`. 159.6k impressions, position 7.8, CTR 0.06%
   — the smallest CTR gap of the top 10, only near the top because of its large volume.
   *Wrong if:* the score's log-volume weighting is over-rewarding raw traffic here relative to
   how fixable the underlying CTR problem actually is.
10. **`content_dd635253d90e`** — `refresh_priority`. 33.3k impressions, position 4.6, CTR 0.02%.
    *Wrong if:* — same as #4, a strong position with near-zero CTR is either a real win waiting
    to happen, or a sign the ranking keyword and the page's actual content have drifted apart.


## 4. Weak picks + leakage check

**Weak pattern in the top 10, stated plainly:** 6 of the top 10 belong to a single client
(`client_19581e27de`), which also happens to be the highest-traffic client in the dataset (max
single-page impressions: 517,109). The score isn't client-normalized, so one very large client
can crowd out equally real, equally actionable opportunities at smaller clients — their CTR gaps
just never reach the same raw score. That's a real weakness of this baseline, not a data
artifact, and it's worth fixing before this becomes a cross-client priority list.

**A second, smaller weak pick:** #9 (`content_91652435f57a`) has the smallest CTR gap of the top
10 (0.2948) and is only in the top 10 because of its large volume — a case where the log-volume
weighting is doing more work than the actual underperformance is.


In [6]:
# Leakage check: confirm the label-related columns never touched the score.
label_cols = {'trend_direction', 'trend_pct', 'is_declining', 'is_declining_label',
              'impressions_last_30d', 'impressions_prev_30d'}
score_inputs = {'days_since_last_update', 'impressions_90d', 'position_tier', 'ctr'}

assert label_cols.isdisjoint(score_inputs), 'leakage: a label-derived column entered the score'
print('Leakage check passed: no label-derived or future-window column feeds the score.')
print(f'Score inputs used: {sorted(score_inputs)}')
print(f"'is_declining' above was computed only to describe Signal 1's bucket table -")
print('it never appears on the right-hand side of the score formula.')


Leakage check passed: no label-derived or future-window column feeds the score.
Score inputs used: ['ctr', 'days_since_last_update', 'impressions_90d', 'position_tier']
'is_declining' above was computed only to describe Signal 1's bucket table -
it never appears on the right-hand side of the score formula.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
